DIMCUSTOMER

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_catalog.retail_gold.dimcustomer
(
    CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID INT,
    CustomerName STRING,
    Email STRING,
    City STRING,
    Address STRING,
    StartDate DATE,
    EndDate DATE,
    IsActive INT
)
USING DELTA
LOCATION 's3://retail-etl-lakehouse/processed/DimCustomer';

SCD-2 Implementation

In [0]:
%sql
-- Expiring old active records where City or Address changed
MERGE INTO gold_catalog.retail_gold.DimCustomer AS target
USING (
    SELECT
        CustomerID,
        INITCAP(TRIM(CustomerName))  AS CustomerName,
        LOWER(TRIM(Email))           AS Email,
        TRIM(City)                   AS City,
        TRIM(Address)                AS Address
    FROM (
        SELECT
            CustomerID,
            CustomerName,
            Email,
            City,
            Address,
            LastUpdated,
            ROW_NUMBER() OVER (
                PARTITION BY CustomerID
                ORDER BY LastUpdated DESC
            ) AS rn
        FROM silver_catalog.retail_silver.silver_customers
        WHERE CustomerID IS NOT NULL
    )
    WHERE rn = 1
) AS source
ON  target.CustomerID = source.CustomerID
AND target.IsActive   = 1

WHEN MATCHED AND (
    target.City    <> source.City OR
    target.Address <> source.Address
)
THEN UPDATE SET
    target.EndDate  = CURRENT_DATE(),
    target.IsActive = 0

WHEN NOT MATCHED
THEN INSERT (
    CustomerID, CustomerName, Email, City, Address, StartDate, EndDate, IsActive
)
VALUES (
    source.CustomerID,
    source.CustomerName,
    source.Email,
    source.City,
    source.Address,
    CURRENT_DATE(),
    DATE('9999-12-31'),
    1
);

-- Insert new active records for customers with City or Address changes
-- Only insert if no active record already exists for this CustomerID with the same City/Address
INSERT INTO gold_catalog.retail_gold.DimCustomer (
    CustomerID, CustomerName, Email, City, Address, StartDate, EndDate, IsActive
)
SELECT
    src.CustomerID,
    src.CustomerName,
    src.Email,
    src.City,
    src.Address,
    CURRENT_DATE() AS StartDate,
    DATE('9999-12-31') AS EndDate,
    1 AS IsActive
FROM (
    SELECT
        CustomerID,
        INITCAP(TRIM(CustomerName))  AS CustomerName,
        LOWER(TRIM(Email))           AS Email,
        TRIM(City)                   AS City,
        TRIM(Address)                AS Address
    FROM (
        SELECT
            CustomerID,
            CustomerName,
            Email,
            City,
            Address,
            LastUpdated,
            ROW_NUMBER() OVER (
                PARTITION BY CustomerID
                ORDER BY LastUpdated DESC
            ) AS rn
        FROM silver_catalog.retail_silver.silver_customers
        WHERE CustomerID IS NOT NULL
    )
    WHERE rn = 1
) src
INNER JOIN gold_catalog.retail_gold.DimCustomer tgt
    ON src.CustomerID = tgt.CustomerID
    AND tgt.EndDate = CURRENT_DATE()
    AND tgt.IsActive = 0
WHERE (
    tgt.City <> src.City OR
    tgt.Address <> src.Address
)
AND NOT EXISTS (
    SELECT 1
    FROM gold_catalog.retail_gold.DimCustomer existing
    WHERE existing.CustomerID = src.CustomerID
    AND existing.City = src.City
    AND existing.Address = src.Address
    AND existing.IsActive = 1
);

DIMPRODUCT

In [0]:
%sql

CREATE TABLE IF NOT EXISTS gold_catalog.retail_gold.DimProduct
(
    ProductSK BIGINT GENERATED ALWAYS AS IDENTITY,

    ProductID INT,
    ProductName STRING,
    Category STRING,
    UnitPrice DECIMAL(10,2),

    EffectiveDate DATE
)
USING DELTA
LOCATION 's3://retail-etl-lakehouse/processed/DimProduct';

In [0]:
%sql
    
--DIM PRODUCT CDC

MERGE INTO gold_catalog.retail_gold.DimProduct tgt
USING (
    SELECT 
        ProductId,
        ProductName, 
        Category,
        UnitPrice,
        CURRENT_DATE() AS EffectiveDate
    FROM silver_catalog.retail_silver.silver_products
    WHERE ProductID IS NOT NULL
    AND UnitPrice > 0
)AS src
ON tgt.ProductID = src.ProductId

WHEN NOT MATCHED THEN INSERT (
    ProductId, ProductName, Category, UnitPrice, EffectiveDate
)
VALUES(
    src.ProductID,
    src.ProductName,
    src.Category,
    src.UnitPrice,
    src.EffectiveDate
);

DIMSTORE

In [0]:
%sql

CREATE TABLE IF NOT EXISTS gold_catalog.retail_gold.DimStore
(
    StoreSK BIGINT GENERATED ALWAYS AS IDENTITY,

    StoreID INT,
    StoreName STRING,
    Region STRING
)
USING DELTA
LOCATION 's3://retail-etl-lakehouse/processed/DimStore';

In [0]:
%sql
MERGE INTO gold_catalog.retail_gold.DimStore tgt
USING (
    SELECT 
        StoreID,
        StoreName,
        COALESCE(Region,'Unknown') AS Region
    FROM silver_catalog.retail_silver.silver_stores
    WHERE StoreID IS NOT NULL
) AS src
ON tgt.StoreID = src.StoreID

WHEN NOT MATCHED THEN INSERT(
    StoreID, StoreName, Region
)
VALUES(
    src.StoreID, src.StoreName, src.Region
);

FACTSALES

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_catalog.retail_gold.FactSales
(
    SalesSK BIGINT,

    TransactionID INT,

    CustomerSK BIGINT,
    ProductSK BIGINT,
    StoreSK BIGINT,

    Quantity INT,
    Amount DECIMAL(10,2),

    TxnDate DATE
)
USING DELTA
LOCATION 's3://retail-etl-lakehouse/processed/FactSales';

In [0]:
%sql
MERGE INTO gold_catalog.retail_gold.FactSales AS target
USING (
    SELECT
        ROW_NUMBER() OVER (ORDER BY s.TransactionID) AS SalesSK,
        s.TransactionID,
        c.CustomerSK,
        p.ProductSK,
        st.StoreSK,
        s.Quantity,
        ROUND(s.Quantity * p.UnitPrice, 2) AS Amount,
        s.TxnDate
    FROM (
        SELECT
            TransactionID,
            CustomerID,
            ProductID,
            StoreID,
            Quantity,
            TxnDate,
            ROW_NUMBER() OVER (
                PARTITION BY TransactionID
                ORDER BY TxnDate ASC
            ) AS rn
        FROM silver_catalog.retail_silver.silver_sales
        WHERE TransactionID IS NOT NULL
          AND Quantity > 0
    ) s
    INNER JOIN gold_catalog.retail_gold.DimCustomer c
        ON s.CustomerID = c.CustomerID
       AND c.IsActive = 1
    LEFT JOIN gold_catalog.retail_gold.DimProduct p
        ON s.ProductID = p.ProductID
    LEFT JOIN gold_catalog.retail_gold.DimStore st
        ON s.StoreID = st.StoreID
    WHERE s.rn = 1
) AS source
ON target.TransactionID = source.TransactionID

WHEN NOT MATCHED
THEN INSERT (
    SalesSK, TransactionID, CustomerSK, ProductSK, StoreSK,
    Quantity, Amount, TxnDate
)
VALUES (
    source.SalesSK,
    source.TransactionID,
    source.CustomerSK,
    source.ProductSK,
    source.StoreSK,
    source.Quantity,
    source.Amount,
    source.TxnDate
);